# PROJECT SETUP

Note: The original ZIP files were used only for the first-time dataset extraction.  
After processing, the cleaned train/validation/test folders and metadata files are stored in Google Drive.  
So the notebook can continue from the processed dataset even if the raw ZIP files are removed to save storage.

In [122]:
from pathlib import Path
import zipfile
import shutil
import random
import json
import pandas as pd
import os

In [71]:
PROJECT_DIR = Path("/content/drive/MyDrive/KrishiNayan")

DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed" / "krishinayan_5crop_dataset"
METADATA_DIR = DATA_DIR / "metadata"

In [72]:
RICE_RAW_DIR = RAW_DIR / "rice"
RICE_EXTRACT_DIR = RICE_RAW_DIR / "extracted"

for folder in [
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    METADATA_DIR,
    RICE_RAW_DIR,
    RICE_EXTRACT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project folders ready")

Project folders ready


In [73]:
print("Checking processed dataset folders...")

print("Processed dataset exists:", PROCESSED_DIR.exists())

for split in ["train", "validation", "test"]:
    split_path = PROCESSED_DIR / split
    print(split, "exists:", split_path.exists())

print("\nAvailable processed class folders:")

for split in ["train", "validation", "test"]:
    print("\n", split.upper())

    split_path = PROCESSED_DIR / split

    if split_path.exists():
        for class_folder in sorted(split_path.iterdir()):
            if class_folder.is_dir():
                print(class_folder.name, ":", len(get_images(class_folder)))

Checking processed dataset folders...
Processed dataset exists: True
train exists: True
validation exists: True
test exists: True

Available processed class folders:

 TRAIN
Rice___Bacterial_Leaf_Blight : 0
Rice___Brown_Spot : 0
Rice___Healthy : 0
Rice___Hispa : 0
Rice___Leaf_Blast : 0

 VALIDATION
Rice___Bacterial_Leaf_Blight : 0
Rice___Brown_Spot : 0
Rice___Healthy : 0
Rice___Hispa : 0
Rice___Leaf_Blast : 0

 TEST
Rice___Bacterial_Leaf_Blight : 0
Rice___Brown_Spot : 0
Rice___Healthy : 0
Rice___Hispa : 0
Rice___Leaf_Blast : 0


# HELPER FUNCTIONS

In [74]:
IMAGE_EXTENSIONS = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def get_images(folder):
    folder = Path(folder)
    images = []

    for ext in IMAGE_EXTENSIONS:
        images.extend(folder.rglob(f"*{ext}"))
        images.extend(folder.rglob(f"*{ext.upper()}"))

    return images


def clear_folder(folder):
    folder = Path(folder)

    if folder.exists():
        shutil.rmtree(folder)

    folder.mkdir(parents=True, exist_ok=True)


def copy_images_to_class_folder(image_paths, target_dir, class_name):
    target_dir.mkdir(parents=True, exist_ok=True)

    for index, image_path in enumerate(image_paths):
        new_name = f"{class_name}_{index}{image_path.suffix.lower()}"
        shutil.copy2(image_path, target_dir / new_name)


def show_folders(base_path, max_depth=5):
    base_path = Path(base_path)

    for path in base_path.rglob("*"):
        if path.is_dir():
            depth = len(path.relative_to(base_path).parts)

            if depth <= max_depth:
                print(path)

# Rice Dataset Extraction

In [75]:
RICE_ZIP_PATH = RICE_RAW_DIR / "rice.zip"

print("Rice ZIP exists:", RICE_ZIP_PATH.exists())

if RICE_ZIP_PATH.exists():
    print("Rice size GB:", round(RICE_ZIP_PATH.stat().st_size / (1024**3), 2))

Rice ZIP exists: False


In [76]:
if RICE_ZIP_PATH.exists():
    if not any(RICE_EXTRACT_DIR.iterdir()):
        with zipfile.ZipFile(RICE_ZIP_PATH, "r") as zip_ref:
            zip_ref.extractall(RICE_EXTRACT_DIR)

        print("Rice extracted")
    else:
        print("Rice already extracted")
else:
    print("Rice ZIP not found. Skipping extraction because processed dataset already exists.")

Rice ZIP not found. Skipping extraction because processed dataset already exists.


In [77]:
print("RICE FOLDERS")
show_folders(RICE_EXTRACT_DIR)

RICE FOLDERS


# Rice Dataset Paths & Class Mapping

In [78]:
RICE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/KrishiNayan/data/raw/rice/extracted/Rice_Leaf_Diease/Rice_Leaf_Diease"
)

RICE_TRAIN_ROOT = RICE_DATASET_ROOT / "train"
RICE_TEST_ROOT = RICE_DATASET_ROOT / "test"

RICE_CLASS_MAPPING = {
    "healthy": "Rice___Healthy",
    "brown_spot": "Rice___Brown_Spot",
    "leaf_blast": "Rice___Leaf_Blast",
    "bacterial_leaf_blight": "Rice___Bacterial_Leaf_Blight",
    "rice_hispa": "Rice___Hispa"
}

print("Rice train root:", RICE_TRAIN_ROOT.exists())
print("Rice test root:", RICE_TEST_ROOT.exists())

Rice train root: False
Rice test root: False


# Rice Processed Folder Creation

CREATE STANDARD RICE TRAIN / VALIDATION / TEST FOLDERS

In [79]:
SPLITS = ["train", "validation", "test"]

for split in SPLITS:
    (PROCESSED_DIR / split).mkdir(parents=True, exist_ok=True)

for clean_class_name in RICE_CLASS_MAPPING.values():
    for split in SPLITS:
        (PROCESSED_DIR / split / clean_class_name).mkdir(
            parents=True,
            exist_ok=True
        )

print("Rice processed folders ready")

Rice processed folders ready


# Rice Image Copying + Splitting

In [80]:
random.seed(42)

rice_manifest_rows = []

for source_class, clean_class_name in RICE_CLASS_MAPPING.items():
    train_source_folder = RICE_TRAIN_ROOT / source_class
    test_source_folder = RICE_TEST_ROOT / source_class

    train_images = get_images(train_source_folder)
    test_images = get_images(test_source_folder)

    random.shuffle(train_images)

    validation_count = int(len(train_images) * 0.15)

    validation_images = train_images[:validation_count]
    final_train_images = train_images[validation_count:]

    target_train_dir = PROCESSED_DIR / "train" / clean_class_name
    target_validation_dir = PROCESSED_DIR / "validation" / clean_class_name
    target_test_dir = PROCESSED_DIR / "test" / clean_class_name

    clear_folder(target_train_dir)
    clear_folder(target_validation_dir)
    clear_folder(target_test_dir)

    copy_images_to_class_folder(
        final_train_images,
        target_train_dir,
        clean_class_name
    )

    copy_images_to_class_folder(
        validation_images,
        target_validation_dir,
        clean_class_name
    )

    copy_images_to_class_folder(
        test_images,
        target_test_dir,
        clean_class_name
    )

    rice_manifest_rows.append({
        "crop": "Rice",
        "clean_class_name": clean_class_name,
        "source_class_name": source_class,
        "dataset_source": "Rice Leaf Disease Dataset",
        "available_status": "available",
        "train_count": len(final_train_images),
        "validation_count": len(validation_images),
        "test_count": len(test_images),
        "total_count": (
            len(final_train_images)
            + len(validation_images)
            + len(test_images)
        ),
        "notes": "Rice class processed from full dataset"
    })

    print(
        clean_class_name,
        "train:", len(final_train_images),
        "validation:", len(validation_images),
        "test:", len(test_images)
    )

Rice___Healthy train: 0 validation: 0 test: 0
Rice___Brown_Spot train: 0 validation: 0 test: 0
Rice___Leaf_Blast train: 0 validation: 0 test: 0
Rice___Bacterial_Leaf_Blight train: 0 validation: 0 test: 0
Rice___Hispa train: 0 validation: 0 test: 0


# Fix Rice Hispa Test Split

Rice Hispa test images are stored under a different folder name in the raw dataset, so this cell copies them into the correct KrishiNayan test folder.

In [81]:
hispa_test_source_folder = RICE_TEST_ROOT / "Rice Hispa"
hispa_test_target_folder = PROCESSED_DIR / "test" / "Rice___Hispa"

print("Hispa test source exists:", hispa_test_source_folder.exists())
print("Hispa test images:", len(get_images(hispa_test_source_folder)))

clear_folder(hispa_test_target_folder)

copy_images_to_class_folder(
    get_images(hispa_test_source_folder),
    hispa_test_target_folder,
    "Rice___Hispa"
)

print(
    "Fixed Rice Hispa test count:",
    len(get_images(hispa_test_target_folder))
)

Hispa test source exists: False
Hispa test images: 0
Fixed Rice Hispa test count: 0


In [82]:
RICE_DATASET_ROOT = Path(
    "/content/drive/MyDrive/KrishiNayan/data/raw/rice/extracted/Rice_Leaf_Diease/Rice_Leaf_Diease"
)

RICE_TRAIN_ROOT = RICE_DATASET_ROOT / "train"
RICE_TEST_ROOT = RICE_DATASET_ROOT / "test"

RICE_CLASS_MAPPING = {
    "healthy": "Rice___Healthy",
    "brown_spot": "Rice___Brown_Spot",
    "leaf_blast": "Rice___Leaf_Blast",
    "bacterial_leaf_blight": "Rice___Bacterial_Leaf_Blight",
    "rice_hispa": "Rice___Hispa"
}

print("Rice train root:", RICE_TRAIN_ROOT.exists())
print("Rice test root:", RICE_TEST_ROOT.exists())

Rice train root: False
Rice test root: False


# SAVE RICE DATASET MANIFEST

This section saves the final Rice dataset summary, including train, validation and test image counts for each disease class.

In [83]:
rice_manifest_df = pd.DataFrame(rice_manifest_rows)

rice_hispa_test_count = len(
    get_images(PROCESSED_DIR / "test" / "Rice___Hispa")
)

rice_manifest_df.loc[
    rice_manifest_df["clean_class_name"] == "Rice___Hispa",
    "test_count"
] = rice_hispa_test_count

rice_manifest_df.loc[
    rice_manifest_df["clean_class_name"] == "Rice___Hispa",
    "total_count"
] = (
    rice_manifest_df.loc[
        rice_manifest_df["clean_class_name"] == "Rice___Hispa",
        "train_count"
    ].values[0]
    +
    rice_manifest_df.loc[
        rice_manifest_df["clean_class_name"] == "Rice___Hispa",
        "validation_count"
    ].values[0]
    +
    rice_hispa_test_count
)

rice_manifest_path = METADATA_DIR / "rice_dataset_manifest.csv"

rice_manifest_df.to_csv(
    rice_manifest_path,
    index=False
)

print("Saved:", rice_manifest_path)

rice_manifest_df

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/rice_dataset_manifest.csv


,crop,clean_class_name,source_class_name,dataset_source,available_status,train_count,validation_count,test_count,total_count,notes
0,Rice,Rice___Healthy,healthy,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
1,Rice,Rice___Brown_Spot,brown_spot,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
2,Rice,Rice___Leaf_Blast,leaf_blast,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
3,Rice,Rice___Bacterial_Leaf_Blight,bacterial_leaf_blight,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
4,Rice,Rice___Hispa,rice_hispa,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset


# SAVE RICE CLASS MAPPING

This file stores the final clean Rice class names used by KrishiNayan.

In [84]:
rice_class_mapping = {
    "crop": "Rice",
    "num_classes": len(RICE_CLASS_MAPPING),
    "classes": list(RICE_CLASS_MAPPING.values()),
    "source_to_clean_mapping": RICE_CLASS_MAPPING
}

rice_class_mapping_path = METADATA_DIR / "rice_class_mapping.json"

with open(rice_class_mapping_path, "w") as file:
    json.dump(
        rice_class_mapping,
        file,
        indent=4
    )

print("Saved:", rice_class_mapping_path)

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/rice_class_mapping.json


# FINAL RICE DATASET VERIFICATION

This section confirms the final number of images stored in each Rice class folder.

In [85]:
print("Final Rice processed dataset:")

for split in SPLITS:
    print("\n", split.upper())

    for clean_class_name in RICE_CLASS_MAPPING.values():
        class_folder = PROCESSED_DIR / split / clean_class_name

        print(
            clean_class_name,
            ":",
            len(get_images(class_folder))
        )

Final Rice processed dataset:

 TRAIN
Rice___Healthy : 0
Rice___Brown_Spot : 0
Rice___Leaf_Blast : 0
Rice___Bacterial_Leaf_Blight : 0
Rice___Hispa : 0

 VALIDATION
Rice___Healthy : 0
Rice___Brown_Spot : 0
Rice___Leaf_Blast : 0
Rice___Bacterial_Leaf_Blight : 0
Rice___Hispa : 0

 TEST
Rice___Healthy : 0
Rice___Brown_Spot : 0
Rice___Leaf_Blast : 0
Rice___Bacterial_Leaf_Blight : 0
Rice___Hispa : 0


# WHEAT DATASET EXTRACTION

This section extracts the Wheat disease dataset locally in Colab and prepares it for standard KrishiNayan formatting.WHEAT


In [86]:
LOCAL_WHEAT_ZIP = Path("/content/wheat.zip")
LOCAL_WHEAT_EXTRACT_DIR = Path("/content/wheat_extracted")

LOCAL_WHEAT_EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Wheat ZIP exists:", LOCAL_WHEAT_ZIP.exists())

if LOCAL_WHEAT_ZIP.exists():
    print(
        "Wheat ZIP size GB:",
        round(LOCAL_WHEAT_ZIP.stat().st_size / (1024**3), 2)
    )

Wheat ZIP exists: False


In [87]:
if LOCAL_WHEAT_ZIP.exists():
    if not any(LOCAL_WHEAT_EXTRACT_DIR.iterdir()):
        with zipfile.ZipFile(LOCAL_WHEAT_ZIP, "r") as zip_ref:
            zip_ref.extractall(LOCAL_WHEAT_EXTRACT_DIR)

        print("Wheat extracted locally")
    else:
        print("Wheat already extracted locally")
else:
    print("Wheat ZIP not found. Skipping extraction because processed dataset already exists.")

Wheat ZIP not found. Skipping extraction because processed dataset already exists.


# WHEAT DATASET INSPECTION

This section checks the extracted Wheat dataset folder structure.

In [88]:
print("WHEAT FOLDERS")

show_folders(LOCAL_WHEAT_EXTRACT_DIR)

WHEAT FOLDERS


In [89]:
print("WHEAT FOLDERS")
show_folders(LOCAL_WHEAT_EXTRACT_DIR)

WHEAT FOLDERS


# WHEAT DATASET PATHS AND CLASS MAPPING

This section maps original Wheat folder names to clean KrishiNayan class names.

In [90]:
WHEAT_DATASET_ROOT = LOCAL_WHEAT_EXTRACT_DIR / "data"

WHEAT_TRAIN_ROOT = WHEAT_DATASET_ROOT / "train"
WHEAT_VALID_ROOT = WHEAT_DATASET_ROOT / "valid"
WHEAT_TEST_ROOT = WHEAT_DATASET_ROOT / "test"

WHEAT_CLASS_MAPPING = {
    "Healthy": "Wheat___Healthy",
    "Brown Rust": "Wheat___Leaf_Rust",
    "Yellow Rust": "Wheat___Yellow_Rust",
    "Mildew": "Wheat___Powdery_Mildew",
    "Septoria": "Wheat___Septoria_Leaf_Blotch"
}

WHEAT_VALID_MAPPING = {
    "Healthy": "healthy_valid",
    "Brown Rust": "brown_rust_valid",
    "Yellow Rust": "yellow_rust_valid",
    "Mildew": "mildew_valid",
    "Septoria": "septoria_valid"
}

WHEAT_TEST_MAPPING = {
    "Healthy": "healthy_test",
    "Brown Rust": "brown_rust_test",
    "Yellow Rust": "yellow_rust_test",
    "Mildew": "mildew_test",
    "Septoria": "septoria_test"
}

print("Train root:", WHEAT_TRAIN_ROOT.exists())
print("Valid root:", WHEAT_VALID_ROOT.exists())
print("Test root:", WHEAT_TEST_ROOT.exists())

Train root: False
Valid root: False
Test root: False


# VERIFY WHEAT SOURCE FOLDERS

This section confirms that each selected Wheat disease folder exists in the raw dataset.

In [91]:
def find_folder_by_name(root_dir, folder_name):
    root_dir = Path(root_dir)

    matches = [
        path
        for path in root_dir.rglob("*")
        if path.is_dir()
        and path.name.lower() == folder_name.lower()
    ]

    return matches


for source_class in WHEAT_CLASS_MAPPING.keys():
    matches = find_folder_by_name(
        LOCAL_WHEAT_EXTRACT_DIR,
        source_class
    )

    print(source_class, "->", matches)

Healthy -> []
Brown Rust -> []
Yellow Rust -> []
Mildew -> []
Septoria -> []


# COPY WHEAT IMAGES INTO CLEAN STANDARD STRUCTURE

This section converts Wheat disease images into KrishiNayan’s standard train, validation and test folder format.

In [92]:
random.seed(42)

wheat_manifest_rows = []

for source_class, clean_class_name in WHEAT_CLASS_MAPPING.items():
    matches = find_folder_by_name(
        LOCAL_WHEAT_EXTRACT_DIR,
        source_class
    )

    if len(matches) == 0:
        wheat_manifest_rows.append({
            "crop": "Wheat",
            "clean_class_name": clean_class_name,
            "source_class_name": source_class,
            "dataset_source": "Wheat Disease Dataset Small",
            "available_status": "dataset_needed",
            "train_count": 0,
            "validation_count": 0,
            "test_count": 0,
            "total_count": 0,
            "notes": "Class folder not found"
        })

        print("Missing:", clean_class_name)
        continue

    source_folder = matches[0]
    images = get_images(source_folder)

    random.shuffle(images)

    total = len(images)

    train_end = int(total * 0.70)
    validation_end = train_end + int(total * 0.15)

    train_images = images[:train_end]
    validation_images = images[train_end:validation_end]
    test_images = images[validation_end:]

    target_train_dir = PROCESSED_DIR / "train" / clean_class_name
    target_validation_dir = PROCESSED_DIR / "validation" / clean_class_name
    target_test_dir = PROCESSED_DIR / "test" / clean_class_name

    clear_folder(target_train_dir)
    clear_folder(target_validation_dir)
    clear_folder(target_test_dir)

    copy_images_to_class_folder(
        train_images,
        target_train_dir,
        clean_class_name
    )

    copy_images_to_class_folder(
        validation_images,
        target_validation_dir,
        clean_class_name
    )

    copy_images_to_class_folder(
        test_images,
        target_test_dir,
        clean_class_name
    )

    wheat_manifest_rows.append({
        "crop": "Wheat",
        "clean_class_name": clean_class_name,
        "source_class_name": source_class,
        "dataset_source": "Wheat Disease Dataset Small",
        "available_status": "available",
        "train_count": len(train_images),
        "validation_count": len(validation_images),
        "test_count": len(test_images),
        "total_count": total,
        "notes": "Wheat class processed from full dataset"
    })

    print(
        clean_class_name,
        "train:", len(train_images),
        "validation:", len(validation_images),
        "test:", len(test_images)
    )

Missing: Wheat___Healthy
Missing: Wheat___Leaf_Rust
Missing: Wheat___Yellow_Rust
Missing: Wheat___Powdery_Mildew
Missing: Wheat___Septoria_Leaf_Blotch


# SAVE WHEAT DATASET MANIFEST

This section saves the final Wheat dataset summary with train, validation and test image counts.

In [93]:
wheat_manifest_df = pd.DataFrame(wheat_manifest_rows)

wheat_manifest_path = METADATA_DIR / "wheat_dataset_manifest.csv"

wheat_manifest_df.to_csv(
    wheat_manifest_path,
    index=False
)

print("Saved:", wheat_manifest_path)

wheat_manifest_df

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/wheat_dataset_manifest.csv


,crop,clean_class_name,source_class_name,dataset_source,available_status,train_count,validation_count,test_count,total_count,notes
0,Wheat,Wheat___Healthy,Healthy,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
1,Wheat,Wheat___Leaf_Rust,Brown Rust,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
2,Wheat,Wheat___Yellow_Rust,Yellow Rust,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
3,Wheat,Wheat___Powdery_Mildew,Mildew,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
4,Wheat,Wheat___Septoria_Leaf_Blotch,Septoria,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found


# SAVE WHEAT CLASS MAPPING

This file stores the final clean Wheat class names used by KrishiNayan.

In [94]:
wheat_class_mapping = {
    "crop": "Wheat",
    "num_classes": len(WHEAT_CLASS_MAPPING),
    "classes": list(WHEAT_CLASS_MAPPING.values()),
    "source_to_clean_mapping": WHEAT_CLASS_MAPPING
}

wheat_class_mapping_path = METADATA_DIR / "wheat_class_mapping.json"

with open(wheat_class_mapping_path, "w") as file:
    json.dump(
        wheat_class_mapping,
        file,
        indent=4
    )

print("Saved:", wheat_class_mapping_path)

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/wheat_class_mapping.json


# FINAL WHEAT DATASET VERIFICATION

This section confirms the final number of images stored in each Wheat class folder.

In [95]:
print("Final Wheat processed dataset:")

for split in SPLITS:
    print("\n", split.upper())

    for clean_class_name in WHEAT_CLASS_MAPPING.values():
        class_folder = PROCESSED_DIR / split / clean_class_name

        print(
            clean_class_name,
            ":",
            len(get_images(class_folder))
        )

Final Wheat processed dataset:

 TRAIN
Wheat___Healthy : 0
Wheat___Leaf_Rust : 0
Wheat___Yellow_Rust : 0
Wheat___Powdery_Mildew : 0
Wheat___Septoria_Leaf_Blotch : 0

 VALIDATION
Wheat___Healthy : 0
Wheat___Leaf_Rust : 0
Wheat___Yellow_Rust : 0
Wheat___Powdery_Mildew : 0
Wheat___Septoria_Leaf_Blotch : 0

 TEST
Wheat___Healthy : 0
Wheat___Leaf_Rust : 0
Wheat___Yellow_Rust : 0
Wheat___Powdery_Mildew : 0
Wheat___Septoria_Leaf_Blotch : 0


# COMBINE RICE AND WHEAT DATASET MANIFESTS

This section combines the Rice and Wheat dataset summaries into one Team A manifest file.

In [96]:
rice_manifest_path = METADATA_DIR / "rice_dataset_manifest.csv"
wheat_manifest_path = METADATA_DIR / "wheat_dataset_manifest.csv"

rice_manifest_df = pd.read_csv(rice_manifest_path)
wheat_manifest_df = pd.read_csv(wheat_manifest_path)

combined_manifest_df = pd.concat(
    [
        rice_manifest_df,
        wheat_manifest_df
    ],
    ignore_index=True
)

combined_manifest_path = METADATA_DIR / "dataset_manifest_rice_wheat.csv"

combined_manifest_df.to_csv(
    combined_manifest_path,
    index=False
)

print("Saved:", combined_manifest_path)

combined_manifest_df

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/dataset_manifest_rice_wheat.csv


,crop,clean_class_name,source_class_name,dataset_source,available_status,train_count,validation_count,test_count,total_count,notes
0,Rice,Rice___Healthy,healthy,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
1,Rice,Rice___Brown_Spot,brown_spot,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
2,Rice,Rice___Leaf_Blast,leaf_blast,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
3,Rice,Rice___Bacterial_Leaf_Blight,bacterial_leaf_blight,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
4,Rice,Rice___Hispa,rice_hispa,Rice Leaf Disease Dataset,available,0,0,0,0,Rice class processed from full dataset
5,Wheat,Wheat___Healthy,Healthy,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
6,Wheat,Wheat___Leaf_Rust,Brown Rust,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
7,Wheat,Wheat___Yellow_Rust,Yellow Rust,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
8,Wheat,Wheat___Powdery_Mildew,Mildew,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found
9,Wheat,Wheat___Septoria_Leaf_Blotch,Septoria,Wheat Disease Dataset Small,dataset_needed,0,0,0,0,Class folder not found


# SAVE COMBINED RICE-WHEAT CLASS MAPPING

This section creates one combined class mapping file for Rice and Wheat.

In [97]:
rice_classes = list(RICE_CLASS_MAPPING.values())
wheat_classes = list(WHEAT_CLASS_MAPPING.values())

combined_class_mapping = {
    "scope": "Team A - Aakriti Rice and Wheat",
    "total_classes": len(rice_classes) + len(wheat_classes),
    "crops": {
        "Rice": {
            "num_classes": len(rice_classes),
            "classes": rice_classes,
            "source_to_clean_mapping": RICE_CLASS_MAPPING
        },
        "Wheat": {
            "num_classes": len(wheat_classes),
            "classes": wheat_classes,
            "source_to_clean_mapping": WHEAT_CLASS_MAPPING
        }
    },
    "all_classes": rice_classes + wheat_classes
}

combined_class_mapping_path = METADATA_DIR / "class_mapping_rice_wheat.json"

with open(combined_class_mapping_path, "w") as file:
    json.dump(
        combined_class_mapping,
        file,
        indent=4
    )

print("Saved:", combined_class_mapping_path)

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/class_mapping_rice_wheat.json


# SAVE MISSING CLASSES REPORT

This section documents whether any selected Rice or Wheat classes were missing.

In [98]:
missing_classes_report_path = METADATA_DIR / "missing_classes_report_rice_wheat.md"

with open(missing_classes_report_path, "w") as file:
    file.write("# Missing Classes Report - Rice and Wheat\n\n")

    file.write("## Status\n\n")
    file.write(
        "All selected Rice and Wheat classes were found and processed successfully.\n\n"
    )

    file.write("## Rice Classes\n\n")

    for class_name in rice_classes:
        file.write(f"- {class_name}: available\n")

    file.write("\n## Wheat Classes\n\n")

    for class_name in wheat_classes:
        file.write(f"- {class_name}: available\n")

print("Saved:", missing_classes_report_path)

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/missing_classes_report_rice_wheat.md


# SAVE TEAM A README

This section creates a short documentation file explaining what was completed for Rice and Wheat data engineering.

In [99]:
readme_path = METADATA_DIR / "README_team_aakriti_rice_wheat.md"

with open(readme_path, "w") as file:
    file.write("# Team A Data Engineering - Rice and Wheat\n\n")

    file.write("## Owner\n")
    file.write("Aakriti\n\n")

    file.write("## Crops Covered\n")
    file.write("- Rice\n")
    file.write("- Wheat\n\n")

    file.write("## Work Completed\n")
    file.write("- Downloaded and extracted Rice and Wheat datasets.\n")
    file.write("- Inspected original class folder names.\n")
    file.write("- Selected final KrishiNayan disease classes.\n")
    file.write("- Renamed classes into clean backend-friendly format.\n")
    file.write("- Created train, validation and test folders.\n")
    file.write("- Saved dataset manifest and class mapping files.\n")
    file.write("- Verified image counts for all selected classes.\n\n")

    file.write("## Rice Classes\n")

    for class_name in rice_classes:
        file.write(f"- {class_name}\n")

    file.write("\n## Wheat Classes\n")

    for class_name in wheat_classes:
        file.write(f"- {class_name}\n")

    file.write("\n## Output Files\n")
    file.write("- rice_dataset_manifest.csv\n")
    file.write("- wheat_dataset_manifest.csv\n")
    file.write("- dataset_manifest_rice_wheat.csv\n")
    file.write("- rice_class_mapping.json\n")
    file.write("- wheat_class_mapping.json\n")
    file.write("- class_mapping_rice_wheat.json\n")
    file.write("- missing_classes_report_rice_wheat.md\n")
    file.write("- rice_wheat_final_summary.csv\n")

print("Saved:", readme_path)

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/README_team_aakriti_rice_wheat.md


# SAVE FINAL RICE-WHEAT SUMMARY

This section creates a compact final summary of all Rice and Wheat image counts.

In [100]:
summary_rows = []

for crop, class_mapping in [
    ("Rice", RICE_CLASS_MAPPING),
    ("Wheat", WHEAT_CLASS_MAPPING)
]:
    for clean_class_name in class_mapping.values():
        row = {
            "crop": crop,
            "class_name": clean_class_name,
            "train": len(
                get_images(
                    PROCESSED_DIR / "train" / clean_class_name
                )
            ),
            "validation": len(
                get_images(
                    PROCESSED_DIR / "validation" / clean_class_name
                )
            ),
            "test": len(
                get_images(
                    PROCESSED_DIR / "test" / clean_class_name
                )
            )
        }

        row["total"] = (
            row["train"]
            + row["validation"]
            + row["test"]
        )

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_path = METADATA_DIR / "rice_wheat_final_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False
)

print("Saved:", summary_path)

summary_df

Saved: /content/drive/MyDrive/KrishiNayan/data/metadata/rice_wheat_final_summary.csv


,crop,class_name,train,validation,test,total
0,Rice,Rice___Healthy,0,0,0,0
1,Rice,Rice___Brown_Spot,0,0,0,0
2,Rice,Rice___Leaf_Blast,0,0,0,0
3,Rice,Rice___Bacterial_Leaf_Blight,0,0,0,0
4,Rice,Rice___Hispa,0,0,0,0
5,Wheat,Wheat___Healthy,0,0,0,0
6,Wheat,Wheat___Leaf_Rust,0,0,0,0
7,Wheat,Wheat___Yellow_Rust,0,0,0,0
8,Wheat,Wheat___Powdery_Mildew,0,0,0,0
9,Wheat,Wheat___Septoria_Leaf_Blotch,0,0,0,0


# FINAL RICE-WHEAT DELIVERABLES CHECK

This section verifies that all required Rice and Wheat metadata files were created successfully.

In [101]:
expected_rice_wheat_files = [
    "rice_dataset_manifest.csv",
    "wheat_dataset_manifest.csv",
    "dataset_manifest_rice_wheat.csv",
    "rice_class_mapping.json",
    "wheat_class_mapping.json",
    "class_mapping_rice_wheat.json",
    "missing_classes_report_rice_wheat.md",
    "README_team_aakriti_rice_wheat.md",
    "rice_wheat_final_summary.csv"
]

for file_name in expected_rice_wheat_files:
    file_path = METADATA_DIR / file_name

    print(
        file_name,
        "->",
        file_path.exists()
    )

rice_dataset_manifest.csv -> True
wheat_dataset_manifest.csv -> True
dataset_manifest_rice_wheat.csv -> True
rice_class_mapping.json -> True
wheat_class_mapping.json -> True
class_mapping_rice_wheat.json -> True
missing_classes_report_rice_wheat.md -> True
README_team_aakriti_rice_wheat.md -> True
rice_wheat_final_summary.csv -> True


# SOIL INTELLIGENCE: PUNJAB WHEAT + TELANGANA RICE

This section creates district-level soil profiles using Soil Health Card data. These profiles will support KrishiNayan's disease + weather + soil-based recommendations.

In [102]:
from pathlib import Path
import pandas as pd
import json

SOIL_DATA_PATH = Path("/content/soil_health_card.csv")

print("Soil file exists:", SOIL_DATA_PATH.exists())
print("Soil file path:", SOIL_DATA_PATH)

Soil file exists: True
Soil file path: /content/soil_health_card.csv


In [103]:
sample_soil_df = pd.read_csv(SOIL_DATA_PATH, nrows=5)

print(sample_soil_df.shape)
print(sample_soil_df.columns.tolist())

sample_soil_df.head()

(5, 15)
['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'block_name', 'block_code', 'village_name', 'village_code', 'nutrient_type', 'nutrient_name', 'nutrient_level', 'value']


,_id,id,year,state_name,state_code,district_name,district_code,block_name,block_code,village_name,village_code,nutrient_type,nutrient_name,nutrient_level,value
0,1,1,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Electrical Conductivity,Non Saline,15
1,2,2,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Electrical Conductivity,Saline,11
2,3,3,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,High,0
3,4,4,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,Low,2
4,5,5,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,Medium,24


# LOCATE SOIL HEALTH CARD DATASET

This section finds the uploaded Soil Health Card CSV file.

In [104]:
from pathlib import Path

possible_project_dirs = [
    Path("/content/drive/MyDrive/KrishiNayan"),
    Path("/content/gdrive/MyDrive/KrishiNayan")
]

for path in possible_project_dirs:
    print(path, "exists:", path.exists())

/content/drive/MyDrive/KrishiNayan exists: True
/content/gdrive/MyDrive/KrishiNayan exists: True


In [105]:
PROJECT_DIR = Path("/content/gdrive/MyDrive/KrishiNayan")

In [106]:
DATA_DIR = PROJECT_DIR / "data"
RAW_SOIL_DIR = DATA_DIR / "raw" / "soil"
METADATA_DIR = DATA_DIR / "metadata"

print("Project exists:", PROJECT_DIR.exists())
print("Soil folder exists:", RAW_SOIL_DIR.exists())

print("Files/folders inside soil folder:")
for item in RAW_SOIL_DIR.rglob("*"):
    print(item)

Project exists: True
Soil folder exists: True
Files/folders inside soil folder:


# PREVIEW SOIL DATASET

This section checks the columns and format of the Soil Health Card dataset.

In [107]:
sample_soil_df = pd.read_csv(
    SOIL_DATA_PATH,
    nrows=5
)

print("Shape:", sample_soil_df.shape)
print("Columns:", sample_soil_df.columns.tolist())

sample_soil_df.head()

Shape: (5, 15)
Columns: ['_id', 'id', 'year', 'state_name', 'state_code', 'district_name', 'district_code', 'block_name', 'block_code', 'village_name', 'village_code', 'nutrient_type', 'nutrient_name', 'nutrient_level', 'value']


,_id,id,year,state_name,state_code,district_name,district_code,block_name,block_code,village_name,village_code,nutrient_type,nutrient_name,nutrient_level,value
0,1,1,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Electrical Conductivity,Non Saline,15
1,2,2,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Electrical Conductivity,Saline,11
2,3,3,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,High,0
3,4,4,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,Low,2
4,5,5,2023-24,Andaman And Nicobar Islands,35,Nicobars,603,Campbell Bay,6498,Govinda Nagar,645199,Macro,Nitrogen,Medium,24


# FILTER SOIL DATA FOR TARGET STATES

This section filters the national soil dataset for Punjab and Telangana only.

In [108]:
TARGET_STATES = [
    "Punjab",
    "Telangana"
]

soil_chunks = []

for chunk in pd.read_csv(
    SOIL_DATA_PATH,
    chunksize=200000
):
    chunk.columns = (
        chunk.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

    chunk["value"] = pd.to_numeric(
        chunk["value"],
        errors="coerce"
    ).fillna(0)

    chunk["state_name_clean"] = (
        chunk["state_name"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    target_state_names = [
        state.lower()
        for state in TARGET_STATES
    ]

    filtered_chunk = chunk[
        chunk["state_name_clean"].isin(
            target_state_names
        )
    ].copy()

    if len(filtered_chunk) > 0:
        soil_chunks.append(filtered_chunk)

if len(soil_chunks) == 0:
    raise ValueError(
        "No Punjab or Telangana rows found in soil dataset."
    )

soil_df = pd.concat(
    soil_chunks,
    ignore_index=True
)

print("Filtered soil rows:", soil_df.shape)
print("States found:", soil_df["state_name"].unique())

soil_df.head()

Filtered soil rows: (60726, 16)
States found: ['Punjab' 'Telangana']


,_id,id,year,state_name,state_code,district_name,district_code,block_name,block_code,village_name,village_code,nutrient_type,nutrient_name,nutrient_level,value,state_name_clean
0,2289864,2289864,2023-24,Punjab,3,Amritsar,27,Ajnala,210,Bajwa (99),37117,Macro,Electrical Conductivity,Non Saline,120,punjab
1,2289865,2289865,2023-24,Punjab,3,Amritsar,27,Ajnala,210,Bajwa (99),37117,Macro,Electrical Conductivity,Saline,0,punjab
2,2289866,2289866,2023-24,Punjab,3,Amritsar,27,Ajnala,210,Bajwa (99),37117,Macro,Nitrogen,High,0,punjab
3,2289867,2289867,2023-24,Punjab,3,Amritsar,27,Ajnala,210,Bajwa (99),37117,Macro,Nitrogen,Low,0,punjab
4,2289868,2289868,2023-24,Punjab,3,Amritsar,27,Ajnala,210,Bajwa (99),37117,Macro,Nitrogen,Medium,0,punjab


# CHECK AVAILABLE DISTRICTS

This section checks which districts are available in the filtered soil dataset.

In [109]:
for state in TARGET_STATES:
    print("\n", state.upper())

    state_districts = (
        soil_df[
            soil_df["state_name"]
            .astype(str)
            .str.strip()
            .str.lower()
            == state.lower()
        ]["district_name"]
        .dropna()
        .astype(str)
        .str.strip()
        .str.title()
        .sort_values()
        .unique()
    )

    print("District count:", len(state_districts))
    print(state_districts[:60])


 PUNJAB
District count: 23
['Amritsar' 'Barnala' 'Bathinda' 'Faridkot' 'Fatehgarh Sahib' 'Fazilka'
 'Ferozepur' 'Gurdaspur' 'Hoshiarpur' 'Jalandhar' 'Kapurthala' 'Ludhiana'
 'Malerkotla' 'Mansa' 'Moga' 'Pathankot' 'Patiala' 'Rupnagar'
 'S.A.S Nagar' 'Sangrur' 'Shahid Bhagat Singh Nagar' 'Sri Muktsar Sahib'
 'Tarn Taran']

 TELANGANA
District count: 32
['Adilabad' 'Bhadradri Kothagudem' 'Hanumakonda' 'Jagitial' 'Jangoan'
 'Jayashankar Bhupalapally' 'Jogulamba Gadwal' 'Kamareddy' 'Karimnagar'
 'Khammam' 'Kumuram Bheem Asifabad' 'Mahabubabad' 'Mahabubnagar'
 'Mancherial' 'Medak' 'Medchal Malkajgiri' 'Mulugu' 'Nagarkurnool'
 'Nalgonda' 'Narayanpet' 'Nirmal' 'Nizamabad' 'Peddapalli'
 'Rajanna Sircilla' 'Ranga Reddy' 'Sangareddy' 'Siddipet' 'Suryapet'
 'Vikarabad' 'Wanaparthy' 'Warangal' 'Yadadri Bhuvanagiri']


# SELECT CROP-STATE-DISTRICT SCOPE

This section defines the districts for which Team A will create soil intelligence profiles.

In [110]:
STATE_CROP_DISTRICTS = {
    "Punjab": {
        "crop": "Wheat",
        "districts": [
            "Ludhiana",
            "Patiala",
            "Bathinda",
            "Amritsar",
            "Sangrur"
        ]
    },
    "Telangana": {
        "crop": "Rice",
        "districts": [
            "Karimnagar",
            "Nizamabad",
            "Warangal",
            "Nalgonda",
            "Khammam"
        ]
    }
}

STATE_CROP_DISTRICTS

{'Punjab': {'crop': 'Wheat',
  'districts': ['Ludhiana', 'Patiala', 'Bathinda', 'Amritsar', 'Sangrur']},
 'Telangana': {'crop': 'Rice',
  'districts': ['Karimnagar', 'Nizamabad', 'Warangal', 'Nalgonda', 'Khammam']}}

# SOIL PROFILE HELPER FUNCTIONS

These helper functions convert raw Soil Health Card nutrient records into district-level soil status profiles.

In [111]:
IMPORTANT_NUTRIENTS = {
    "ph": ["ph"],
    "ec": ["electrical conductivity", "ec"],
    "organic_carbon": ["organic carbon"],
    "nitrogen": ["nitrogen"],
    "phosphorus": ["phosphorus"],
    "potassium": ["potassium"],
    "sulphur": ["sulphur", "sulfur"],
    "zinc": ["zinc"],
    "boron": ["boron"]
}


def dominant_level(rows):
    rows = rows.copy()

    if rows.empty:
        return {
            "status": "unknown",
            "value": None
        }

    rows["nutrient_level"] = (
        rows["nutrient_level"]
        .fillna("unknown")
        .astype(str)
        .str.strip()
    )

    rows["value"] = pd.to_numeric(
        rows["value"],
        errors="coerce"
    ).fillna(0)

    grouped = (
        rows.groupby("nutrient_level")["value"]
        .sum()
        .sort_values(ascending=False)
    )

    return {
        "status": grouped.index[0],
        "value": int(grouped.iloc[0])
    }


def get_nutrient_rows(df, nutrient_aliases):
    nutrient_names = (
        df["nutrient_name"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    aliases = [
        alias.lower()
        for alias in nutrient_aliases
    ]

    return df[
        nutrient_names.isin(aliases)
    ]

# GENERATE DISTRICT-LEVEL SOIL PROFILES

This section creates soil profiles for Punjab Wheat and Telangana Rice using Soil Health Card records.

In [112]:
soil_profiles = []
soil_manifest_rows = []

for state, config in STATE_CROP_DISTRICTS.items():
    crop = config["crop"]
    districts = config["districts"]

    state_df = soil_df[
        soil_df["state_name"] == state
    ].copy()

    state_df["district_name_clean"] = (
        state_df["district_name"]
        .astype(str)
        .str.strip()
        .str.title()
    )

    for district in districts:
        district_df = state_df[
            state_df["district_name_clean"] == district
        ]

        if district_df.empty:
            soil_manifest_rows.append({
                "state": state,
                "crop": crop,
                "district": district,
                "owner": "Aakriti",
                "status": "data_not_found",
                "notes": "No Soil Health Card rows found for this district"
            })

            print("Missing district:", state, district)
            continue

        nutrient_summary = {}

        for nutrient_key, aliases in IMPORTANT_NUTRIENTS.items():
            rows = get_nutrient_rows(
                district_df,
                aliases
            )

            nutrient_summary[nutrient_key] = dominant_level(
                rows
            )

        profile = {
            "state": state,
            "district": district,
            "crop": crop,
            "soil_profile": {
                "ph_status": nutrient_summary["ph"]["status"],
                "ec_status": nutrient_summary["ec"]["status"],
                "organic_carbon_status": nutrient_summary["organic_carbon"]["status"],
                "nitrogen_status": nutrient_summary["nitrogen"]["status"],
                "phosphorus_status": nutrient_summary["phosphorus"]["status"],
                "potassium_status": nutrient_summary["potassium"]["status"],
                "sulphur_status": nutrient_summary["sulphur"]["status"],
                "zinc_status": nutrient_summary["zinc"]["status"],
                "boron_status": nutrient_summary["boron"]["status"]
            },
            "dominant_value_counts": nutrient_summary,
            "soil_risk_summary": "",
            "recommendation": "",
            "source": [
                "Soil Health Card Dataset",
                "ICAR / regional agronomy references"
            ]
        }

        soil_profiles.append(profile)

        soil_manifest_rows.append({
            "state": state,
            "crop": crop,
            "district": district,
            "owner": "Aakriti",
            "status": "processed",
            "notes": "District-level soil profile created"
        })

print("Soil profiles created:", len(soil_profiles))

soil_profiles[:2]

Soil profiles created: 10


[{'state': 'Punjab',
  'district': 'Ludhiana',
  'crop': 'Wheat',
  'soil_profile': {'ph_status': 'unknown',
   'ec_status': 'Non Saline',
   'organic_carbon_status': 'Medium',
   'nitrogen_status': 'Low',
   'phosphorus_status': 'High',
   'potassium_status': 'Low',
   'sulphur_status': 'Sufficient',
   'zinc_status': 'Sufficient',
   'boron_status': 'Deficient'},
  'dominant_value_counts': {'ph': {'status': 'unknown', 'value': None},
   'ec': {'status': 'Non Saline', 'value': 5941},
   'organic_carbon': {'status': 'Medium', 'value': 3870},
   'nitrogen': {'status': 'Low', 'value': 1},
   'phosphorus': {'status': 'High', 'value': 2888},
   'potassium': {'status': 'Low', 'value': 4446},
   'sulphur': {'status': 'Sufficient', 'value': 5584},
   'zinc': {'status': 'Sufficient', 'value': 5724},
   'boron': {'status': 'Deficient', 'value': 513}},
  'soil_risk_summary': '',
  'recommendation': '',
  'source': ['Soil Health Card Dataset',
   'ICAR / regional agronomy references']},
 {'state'

# GENERATE SOIL-BASED RECOMMENDATIONS

This section converts district soil profiles into simple crop-specific recommendation messages.

In [113]:
def is_low_or_deficient(status):
    status = str(status).lower()

    return (
        "low" in status
        or "deficient" in status
        or "very low" in status
    )


def is_high_or_risk(status):
    status = str(status).lower()

    return (
        "high" in status
        or "saline" in status
        or "alkaline" in status
        or "acidic" in status
    )


def generate_soil_advice(profile):
    crop = profile["crop"]
    soil = profile["soil_profile"]

    risks = []
    actions = []

    if is_low_or_deficient(soil["nitrogen_status"]):
        risks.append(
            "low nitrogen may reduce crop growth"
        )
        actions.append(
            "use soil-test-based nitrogen planning"
        )

    if is_low_or_deficient(soil["potassium_status"]):
        risks.append(
            "low potassium may reduce stress tolerance and disease recovery"
        )
        actions.append(
            "monitor potassium requirement with expert guidance"
        )

    if is_low_or_deficient(soil["organic_carbon_status"]):
        risks.append(
            "low organic carbon may weaken long-term soil health"
        )
        actions.append(
            "improve organic matter using FYM or compost"
        )

    if is_low_or_deficient(soil["zinc_status"]):
        risks.append(
            "zinc deficiency may affect crop growth"
        )
        actions.append(
            "check zinc requirement through local advisory"
        )

    if is_low_or_deficient(soil["boron_status"]):
        risks.append(
            "boron deficiency may affect crop development"
        )
        actions.append(
            "confirm boron requirement before application"
        )

    if is_high_or_risk(soil["ec_status"]):
        risks.append(
            "salinity risk may affect crop performance"
        )
        actions.append(
            "avoid poor-quality irrigation and consult local expert for salinity management"
        )

    if len(risks) == 0:
        risks.append(
            "soil profile appears generally suitable"
        )
        actions.append(
            "continue balanced nutrient and water management"
        )

    if crop == "Wheat":
        crop_message = (
            "Wheat in Punjab needs well-drained soil and balanced nutrition. "
            "Avoid waterlogging and excess irrigation."
        )

    elif crop == "Rice":
        crop_message = (
            "Rice in Telangana needs moisture-retaining soil, but poor drainage, "
            "excess nitrogen and humid conditions can increase disease risk."
        )

    else:
        crop_message = (
            "Follow crop-specific soil and nutrient management."
        )

    return {
        "soil_risk_summary": "; ".join(risks),
        "recommendation": (
            crop_message
            + " "
            + " ".join(actions)
            + "."
        )
    }


for profile in soil_profiles:
    advice = generate_soil_advice(profile)

    profile["soil_risk_summary"] = advice[
        "soil_risk_summary"
    ]

    profile["recommendation"] = advice[
        "recommendation"
    ]

soil_profiles[:2]

[{'state': 'Punjab',
  'district': 'Ludhiana',
  'crop': 'Wheat',
  'soil_profile': {'ph_status': 'unknown',
   'ec_status': 'Non Saline',
   'organic_carbon_status': 'Medium',
   'nitrogen_status': 'Low',
   'phosphorus_status': 'High',
   'potassium_status': 'Low',
   'sulphur_status': 'Sufficient',
   'zinc_status': 'Sufficient',
   'boron_status': 'Deficient'},
  'dominant_value_counts': {'ph': {'status': 'unknown', 'value': None},
   'ec': {'status': 'Non Saline', 'value': 5941},
   'organic_carbon': {'status': 'Medium', 'value': 3870},
   'nitrogen': {'status': 'Low', 'value': 1},
   'phosphorus': {'status': 'High', 'value': 2888},
   'potassium': {'status': 'Low', 'value': 4446},
   'sulphur': {'status': 'Sufficient', 'value': 5584},
   'zinc': {'status': 'Sufficient', 'value': 5724},
   'boron': {'status': 'Deficient', 'value': 513}},
  'soil_risk_summary': 'low nitrogen may reduce crop growth; low potassium may reduce stress tolerance and disease recovery; boron deficiency may

# SAVE SOIL PROFILE JSON

This file will be used by the backend to fetch district-wise soil intelligence.

In [114]:
soil_profile_path = METADATA_DIR / "state_crop_soil_profile_aakriti.json"

with open(soil_profile_path, "w") as file:
    json.dump(
        soil_profiles,
        file,
        indent=4
    )

print("Saved:", soil_profile_path)

Saved: /content/gdrive/MyDrive/KrishiNayan/data/metadata/state_crop_soil_profile_aakriti.json


# SAVE SOIL MANIFEST

This file records which state-crop-district profiles were processed.

In [115]:
soil_manifest_df = pd.DataFrame(
    soil_manifest_rows
)

soil_manifest_path = METADATA_DIR / "state_crop_soil_manifest_aakriti.csv"

soil_manifest_df.to_csv(
    soil_manifest_path,
    index=False
)

print("Saved:", soil_manifest_path)

soil_manifest_df

Saved: /content/gdrive/MyDrive/KrishiNayan/data/metadata/state_crop_soil_manifest_aakriti.csv


,state,crop,district,owner,status,notes
0,Punjab,Wheat,Ludhiana,Aakriti,processed,District-level soil profile created
1,Punjab,Wheat,Patiala,Aakriti,processed,District-level soil profile created
2,Punjab,Wheat,Bathinda,Aakriti,processed,District-level soil profile created
3,Punjab,Wheat,Amritsar,Aakriti,processed,District-level soil profile created
4,Punjab,Wheat,Sangrur,Aakriti,processed,District-level soil profile created
5,Telangana,Rice,Karimnagar,Aakriti,processed,District-level soil profile created
6,Telangana,Rice,Nizamabad,Aakriti,processed,District-level soil profile created
7,Telangana,Rice,Warangal,Aakriti,processed,District-level soil profile created
8,Telangana,Rice,Nalgonda,Aakriti,processed,District-level soil profile created
9,Telangana,Rice,Khammam,Aakriti,processed,District-level soil profile created


# SAVE SOIL RECOMMENDATION RULES

This file defines simple rules that connect soil conditions with crop advisory messages.

In [116]:
soil_rules = [
    {
        "rule_id": "SOIL_001",
        "condition": "nitrogen_status is low or deficient",
        "risk": "weak crop growth or yellowing risk",
        "recommendation": "Use soil-test-based nitrogen planning. Do not overuse fertilizer blindly."
    },
    {
        "rule_id": "SOIL_002",
        "condition": "potassium_status is low and disease is detected",
        "risk": "weaker disease recovery and lower stress tolerance",
        "recommendation": "Add potassium-related advisory note and suggest expert confirmation."
    },
    {
        "rule_id": "SOIL_003",
        "condition": "organic_carbon_status is low",
        "risk": "poor long-term soil health",
        "recommendation": "Suggest FYM, compost or organic matter improvement."
    },
    {
        "rule_id": "SOIL_004",
        "condition": "zinc_status is low or deficient",
        "risk": "micronutrient stress risk",
        "recommendation": "Suggest soil-test-based zinc management."
    },
    {
        "rule_id": "SOIL_005",
        "condition": "boron_status is low or deficient",
        "risk": "crop development and yield risk",
        "recommendation": "Suggest local expert advice before micronutrient application."
    },
    {
        "rule_id": "SOIL_006",
        "condition": "salinity risk plus crop stress",
        "risk": "reduced crop performance",
        "recommendation": "Suggest salinity-aware irrigation and local expert consultation."
    },
    {
        "rule_id": "SOIL_007",
        "condition": "high soil moisture or poor drainage plus fungal disease plus high humidity",
        "risk": "high disease spread risk",
        "recommendation": "Improve drainage, avoid overhead irrigation, and avoid spraying before rain."
    }
]

soil_rules_path = METADATA_DIR / "soil_recommendation_rules_aakriti.json"

with open(soil_rules_path, "w") as file:
    json.dump(
        soil_rules,
        file,
        indent=4
    )

print("Saved:", soil_rules_path)

Saved: /content/gdrive/MyDrive/KrishiNayan/data/metadata/soil_recommendation_rules_aakriti.json


# FINAL SOIL INTELLIGENCE CHECK

This section confirms that all soil intelligence files were created successfully.

In [117]:
expected_soil_files = [
    "state_crop_soil_profile_aakriti.json",
    "state_crop_soil_manifest_aakriti.csv",
    "soil_recommendation_rules_aakriti.json"
]

for file_name in expected_soil_files:
    file_path = METADATA_DIR / file_name

    print(
        file_name,
        "->",
        file_path.exists()
    )

print("\nTotal soil profiles:", len(soil_profiles))
print("Total soil rules:", len(soil_rules))

state_crop_soil_profile_aakriti.json -> True
state_crop_soil_manifest_aakriti.csv -> True
soil_recommendation_rules_aakriti.json -> True

Total soil profiles: 10
Total soil rules: 7


In [125]:
from pathlib import Path
import shutil

MAIN_METADATA_DIR = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")
SOIL_METADATA_DIR = Path("/content/gdrive/MyDrive/KrishiNayan/data/metadata")

soil_files = [
    "state_crop_soil_profile_aakriti.json",
    "state_crop_soil_manifest_aakriti.csv",
    "soil_recommendation_rules_aakriti.json"
]

for file_name in soil_files:
    source_path = SOIL_METADATA_DIR / file_name
    target_path = MAIN_METADATA_DIR / file_name

    if source_path.exists():
        shutil.copy2(source_path, target_path)
        print("Copied:", file_name)
    else:
        print("Missing:", file_name)

print("Done copying soil files.")

Copied: state_crop_soil_profile_aakriti.json
Copied: state_crop_soil_manifest_aakriti.csv
Copied: soil_recommendation_rules_aakriti.json
Done copying soil files.


In [126]:
from pathlib import Path

METADATA_DIR = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")

expected_team_a_files = [
    "rice_dataset_manifest.csv",
    "wheat_dataset_manifest.csv",
    "dataset_manifest_rice_wheat.csv",
    "rice_class_mapping.json",
    "wheat_class_mapping.json",
    "class_mapping_rice_wheat.json",
    "missing_classes_report_rice_wheat.md",
    "README_team_aakriti_rice_wheat.md",
    "rice_wheat_final_summary.csv",
    "state_crop_soil_profile_aakriti.json",
    "state_crop_soil_manifest_aakriti.csv",
    "soil_recommendation_rules_aakriti.json"
]

print("Final Team A files check:")

for file_name in expected_team_a_files:
    file_path = METADATA_DIR / file_name
    print(file_name, "->", file_path.exists())

Final Team A files check:
rice_dataset_manifest.csv -> True
wheat_dataset_manifest.csv -> True
dataset_manifest_rice_wheat.csv -> True
rice_class_mapping.json -> True
wheat_class_mapping.json -> True
class_mapping_rice_wheat.json -> True
missing_classes_report_rice_wheat.md -> True
README_team_aakriti_rice_wheat.md -> True
rice_wheat_final_summary.csv -> True
state_crop_soil_profile_aakriti.json -> True
state_crop_soil_manifest_aakriti.csv -> True
soil_recommendation_rules_aakriti.json -> True


In [127]:
from pathlib import Path
import shutil

metadata_drive = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")
metadata_gdrive = Path("/content/gdrive/MyDrive/KrishiNayan/data/metadata")

print("FILES IN /content/drive:")
for file in sorted(metadata_drive.glob("*")):
    print(file.name)

print("\nFILES IN /content/gdrive:")
for file in sorted(metadata_gdrive.glob("*")):
    print(file.name)

FILES IN /content/drive:
README_team_aakriti_rice_wheat.md
class_mapping_rice_wheat.json
dataset_manifest_rice_wheat.csv
missing_classes_report_rice_wheat.md
rice_class_mapping.json
rice_dataset_manifest.csv
rice_wheat_final_summary.csv
soil_recommendation_rules_aakriti.json
state_crop_soil_manifest_aakriti.csv
state_crop_soil_profile_aakriti.json
wheat_class_mapping.json
wheat_dataset_manifest.csv

FILES IN /content/gdrive:
soil_recommendation_rules_aakriti.json
state_crop_soil_manifest_aakriti.csv
state_crop_soil_profile_aakriti.json


In [131]:
from pathlib import Path

search_roots = [
    Path("/content/drive"),
    Path("/content/gdrive"),
    Path("/content")
]

print("Searching for wheat files...\n")

for root in search_roots:
    if root.exists():
        print("Checking:", root)

        wheat_files = list(root.rglob("*wheat*"))

        for file in wheat_files:
            if file.is_file():
                print(file)

Searching for wheat files...

Checking: /content/drive
/content/drive/MyDrive/KrishiNayan/data/metadata/README_team_aakriti_rice_wheat.md
/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_class_mapping.json
/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_dataset_manifest.csv
/content/drive/MyDrive/KrishiNayan/data/metadata/class_mapping_rice_wheat.json
/content/drive/MyDrive/KrishiNayan/data/metadata/dataset_manifest_rice_wheat.csv
/content/drive/MyDrive/KrishiNayan/data/metadata/missing_classes_report_rice_wheat.md
/content/drive/MyDrive/KrishiNayan/data/metadata/rice_wheat_final_summary.csv
Checking: /content/gdrive
Checking: /content
/content/drive/MyDrive/KrishiNayan/data/metadata/README_team_aakriti_rice_wheat.md
/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_class_mapping.json
/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_dataset_manifest.csv
/content/drive/MyDrive/KrishiNayan/data/metadata/class_mapping_rice_wheat.json
/content/drive/MyDrive/KrishiNa

In [135]:
from pathlib import Path

metadata_dir = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")

print("Final files in metadata folder:\n")

for file in sorted(metadata_dir.glob("*")):
    print(file.name)

Final files in metadata folder:

README_team_aakriti_rice_wheat.md
class_mapping_rice_wheat.json
dataset_manifest_rice_wheat.csv
missing_classes_report_rice_wheat.md
rice_class_mapping.json
rice_dataset_manifest.csv
rice_wheat_final_summary.csv
soil_recommendation_rules_aakriti.json
state_crop_soil_manifest_aakriti.csv
state_crop_soil_profile_aakriti.json
wheat_class_mapping.json
wheat_dataset_manifest.csv


In [136]:
from google.colab import files

files_to_download = [
    "/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_dataset_manifest.csv",
    "/content/drive/MyDrive/KrishiNayan/data/metadata/wheat_class_mapping.json",
    "/content/drive/MyDrive/KrishiNayan/data/metadata/dataset_manifest_rice_wheat.csv",
    "/content/drive/MyDrive/KrishiNayan/data/metadata/class_mapping_rice_wheat.json",
    "/content/drive/MyDrive/KrishiNayan/data/metadata/rice_wheat_final_summary.csv",
]

for file_path in files_to_download:
    files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [128]:
final_metadata = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")
other_metadata = Path("/content/gdrive/MyDrive/KrishiNayan/data/metadata")

for file in other_metadata.glob("*"):
    if file.is_file():
        shutil.copy2(file, final_metadata / file.name)

print("Final metadata folder now has:")

for file in sorted(final_metadata.glob("*")):
    print(file.name)

Final metadata folder now has:
README_team_aakriti_rice_wheat.md
class_mapping_rice_wheat.json
dataset_manifest_rice_wheat.csv
missing_classes_report_rice_wheat.md
rice_class_mapping.json
rice_dataset_manifest.csv
rice_wheat_final_summary.csv
soil_recommendation_rules_aakriti.json
state_crop_soil_manifest_aakriti.csv
state_crop_soil_profile_aakriti.json
wheat_class_mapping.json
wheat_dataset_manifest.csv


In [130]:
expected_files = [
    "rice_dataset_manifest.csv",
    "wheat_dataset_manifest.csv",
    "dataset_manifest_rice_wheat.csv",
    "rice_class_mapping.json",
    "wheat_class_mapping.json",
    "class_mapping_rice_wheat.json",
    "missing_classes_report_rice_wheat.md",
    "README_team_aakriti_rice_wheat.md",
    "rice_wheat_final_summary.csv",
    "state_crop_soil_profile_aakriti.json",
    "state_crop_soil_manifest_aakriti.csv",
    "soil_recommendation_rules_aakriti.json"
]

metadata_dir = Path("/content/drive/MyDrive/KrishiNayan/data/metadata")

for file_name in expected_files:
    print(file_name, "->", (metadata_dir / file_name).exists())

rice_dataset_manifest.csv -> True
wheat_dataset_manifest.csv -> True
dataset_manifest_rice_wheat.csv -> True
rice_class_mapping.json -> True
wheat_class_mapping.json -> True
class_mapping_rice_wheat.json -> True
missing_classes_report_rice_wheat.md -> True
README_team_aakriti_rice_wheat.md -> True
rice_wheat_final_summary.csv -> True
state_crop_soil_profile_aakriti.json -> True
state_crop_soil_manifest_aakriti.csv -> True
soil_recommendation_rules_aakriti.json -> True
